In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.sql.functions import col
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType, TimestampType

In [2]:
#/////////////////////////////////////////////////////////
# preprocessing data.csv
#////////////////////////////////////////////////////////

In [3]:
spark = SparkSession.builder.appName("MusicRecommender").getOrCreate()

data = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://hadoop-namenode-1:8020/data/data.csv")

data = data.dropna()

columns_to_conver_double = ['valence', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness',
                  'loudness', 'speechiness', 'tempo']
for col_name in columns_to_conver_double:
    data = data.withColumn(col_name, col(col_name).cast(DoubleType()))

columns_to_conver_int = ['year','duration_ms','explicit','key','mode','popularity']
for col_name in columns_to_conver_int:
    data = data.withColumn(col_name, col(col_name).cast(IntegerType()))

columns_to_conver_timestamp = ['release_date']
for col_name in columns_to_conver_timestamp:
    data = data.withColumn(col_name, col(col_name).cast(TimestampType()))

data.printSchema()

root
 |-- valence: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- artists: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- energy: double (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- id: string (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- release_date: timestamp (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)



In [4]:
#/////////////////////////////////////////////////////////////////
# preprocessing data_by_genres.csv
#///////////////////////////////////////////////////////////////////

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans

spark = SparkSession.builder.appName("MusicRecommender").getOrCreate()
data_by_genres = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://hadoop-namenode-1:8020/data/data_by_genres.csv")

data_by_genres = data_by_genres.dropna()

In [6]:
#///////////////////////////////////////////////////////////////
# user inputs
#//////////////////////////////////////////////////////////////

In [7]:
inputs_1 = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://hadoop-namenode-1:8020/data/inputs_1.csv")
inputs_1.show(5) # musisque piano / chill ( sans parole)

inputs_2 = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://hadoop-namenode-1:8020/data/inputs_2.csv")
inputs_2.show(5) # musique K-pop

inputs_3 = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://hadoop-namenode-1:8020/data/inputs_3.csv")
inputs_3.show(5) # musique Rap-us

+-------+----+------------+--------------------+------------------+-----------+------------------+--------+--------------------+----------------+---+--------+--------+----+--------------------+----------+-------------------+-----------+------------------+
|valence|year|acousticness|             artists|      danceability|duration_ms|            energy|explicit|                  id|instrumentalness|key|liveness|loudness|mode|                name|popularity|       release_date|speechiness|             tempo|
+-------+----+------------+--------------------+------------------+-----------+------------------+--------+--------------------+----------------+---+--------+--------+----+--------------------+----------+-------------------+-----------+------------------+
|  0.898|1994|       0.321|['Jason Weaver', ...|0.6890000000000001|     170880|0.5329999999999999|       0|0qxtQ8rf3W1nId3D2...|         7.37E-5|  6|  0.0958| -14.205|   1|"I Just Can't Wai...|        69|1994-01-01 00:00:00|     0.0

In [8]:
features = [
    'acousticness', 'danceability', 'duration_ms', 'energy',
    'instrumentalness', 'liveness', 'loudness', 'speechiness',
    'tempo', 'valence', 'popularity'
]

data = data.dropna(subset=features)

assembler = VectorAssembler(inputCols=features, outputCol="features")
data = assembler.transform(data)

scaler = StandardScaler(
    inputCol="features", 
    outputCol="scaledFeatures",
    withMean=True, 
    withStd=True
)
scaler_model = scaler.fit(data)
data = scaler_model.transform(data)

k = 1000
kmeans = KMeans(
    k=k, 
    seed=42, 
    featuresCol="scaledFeatures", 
    predictionCol="cluster"
)
kmeans_model = kmeans.fit(data)
data = kmeans_model.transform(data)
data.createOrReplaceTempView("all_tracks")

In [9]:
inputs = inputs_1.dropna(subset=features)

inputs = assembler.transform(inputs)

inputs = scaler_model.transform(inputs)

inputs = kmeans_model.transform(inputs)

# vues temporaires SQL
inputs.createOrReplaceTempView("user_tracks")

# 2 clusters
top_clusters = spark.sql("""
SELECT cluster, COUNT(*) AS count
FROM user_tracks
GROUP BY cluster
ORDER BY count DESC
LIMIT 2
""" )
top_clusters.createOrReplaceTempView("top_clusters")

# tracks -> cluster
recommendations = spark.sql("""
SELECT DISTINCT a.*
FROM all_tracks a
JOIN top_clusters t ON a.cluster = t.cluster
WHERE a.id NOT IN (SELECT id FROM user_tracks)
ORDER BY a.popularity DESC
LIMIT 10
""" )


print("Clusters des musiques en entrée de l'utilisateur :")
inputs.select("name", "artists", "cluster","instrumentalness") \
      .distinct() \
      .orderBy("cluster") \
      .show(truncate=False)

print("Recommandations :")
recommendations.select("name", "artists", "cluster", "instrumentalness") \
               .show(truncate=False)


Clusters des musiques en entrée de l'utilisateur :
+--------------------------------------------------------------------------+----------------------------------------------------+-------+----------------+
|name                                                                      |artists                                             |cluster|instrumentalness|
+--------------------------------------------------------------------------+----------------------------------------------------+-------+----------------+
|Nocturne No.4 In F, Op.15 No.1                                            |['Frédéric Chopin', 'Daniel Barenboim']             |3      |0.92            |
|Nocturne in E minor, Op. 72, No. 1                                        |['Frédéric Chopin', 'Janusz Olejniczak']            |387    |0.882           |
|Welcome To Jurassic Park                                                  |['John Williams']                                   |614    |0.882           |
|Una Mattina       

In [10]:
inputs = inputs_2.dropna(subset=features)

inputs = assembler.transform(inputs)

inputs = scaler_model.transform(inputs)

inputs = kmeans_model.transform(inputs)

inputs.createOrReplaceTempView("user_tracks")

top_clusters = spark.sql("""
SELECT cluster, COUNT(*) AS count
FROM user_tracks
GROUP BY cluster
ORDER BY count DESC
LIMIT 2
""" )
top_clusters.createOrReplaceTempView("top_clusters")

recommendations = spark.sql("""
SELECT DISTINCT a.*
FROM all_tracks a
JOIN top_clusters t ON a.cluster = t.cluster
WHERE a.id NOT IN (SELECT id FROM user_tracks)
ORDER BY a.popularity DESC
LIMIT 10
""" )


print("Clusters des musiques en entrée de l'utilisateur :")
inputs.select("name", "artists", "cluster","instrumentalness") \
      .distinct() \
      .orderBy("cluster") \
      .show(truncate=False)

print("Recommandations :")
recommendations.select("name", "artists", "cluster", "instrumentalness") \
               .show(truncate=False)


Clusters des musiques en entrée de l'utilisateur :
+--------------------------+--------------+-------+----------------+
|name                      |artists       |cluster|instrumentalness|
+--------------------------+--------------+-------+----------------+
|Lovesick Girls            |['BLACKPINK'] |32     |0.0             |
|BOOMBAYAH                 |['BLACKPINK'] |227    |1.02E-6         |
|CHEER UP                  |['TWICE']     |447    |0.0             |
|Gangnam Style (강남스타일)|['PSY']       |617    |0.0             |
|Back Door                 |['Stray Kids']|682    |0.0             |
+--------------------------+--------------+-------+----------------+

Recommandations :
+----------------------+--------------------+-------+----------------+
|name                  |artists             |cluster|instrumentalness|
+----------------------+--------------------+-------+----------------+
|Santa Tell Me         |['Ariana Grande']   |447    |0.0             |
|Back In Black         |['AC/D

In [11]:
inputs = inputs_3.dropna(subset=features)

inputs = assembler.transform(inputs)

inputs = scaler_model.transform(inputs)

inputs = kmeans_model.transform(inputs)

inputs.createOrReplaceTempView("user_tracks")

top_clusters = spark.sql("""
SELECT cluster, COUNT(*) AS count
FROM user_tracks
GROUP BY cluster
ORDER BY count DESC
LIMIT 2
""" )
top_clusters.createOrReplaceTempView("top_clusters")

recommendations = spark.sql("""
SELECT DISTINCT a.*
FROM all_tracks a
JOIN top_clusters t ON a.cluster = t.cluster
WHERE a.id NOT IN (SELECT id FROM user_tracks)
ORDER BY a.popularity DESC
LIMIT 10
""" )


print("Clusters des musiques en entrée de l'utilisateur :")
inputs.select("name", "artists", "cluster","instrumentalness") \
      .distinct() \
      .orderBy("cluster") \
      .show(truncate=False)

print("Recommandations :")
recommendations.select("name", "artists", "cluster", "instrumentalness") \
               .show(truncate=False)


Clusters des musiques en entrée de l'utilisateur :
+---------------------------------------------------------+-----------------------------------------------------+-------+----------------+
|name                                                     |artists                                              |cluster|instrumentalness|
+---------------------------------------------------------+-----------------------------------------------------+-------+----------------+
|Love Me                                                  |['Lil Wayne', 'Drake', 'Future']                     |32     |0.0             |
|Rap God                                                  |['Eminem']                                           |88     |0.0             |
|Suit & Tie (feat. JAY Z) (feat. Jay-Z) - [Radio Edit]    |['Justin Timberlake', 'JAY-Z']                       |378    |1.37E-6         |
|F**kin' Problems (feat. Drake, 2 Chainz & Kendrick Lamar)|['A$AP Rocky', 'Drake', '2 Chainz', 'Kendrick Lamar']|68